# Setup

In [1]:
# install personal package for use below, type the below into the docker terminal
# pip install /tf/pyGroupSequentialDesigns/

In [2]:
# higher resolution graphs
%config InlineBackend.figure_format='retina'

## Published package imports

In [3]:
# imports for study design step (Step 1)
import numpy as np
import pandas as pd
from scipy import stats

# imports for GP regression (Step 3)
import gpflow

# imports for Bayes opt (Step 4-6)
import trieste
from trieste.space import Box
from trieste.models.gpflow.models import GaussianProcessRegression
import tensorflow as tf

/usr/local/lib/python3.11/dist-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
/usr/local/lib/python3.11/dist-packages/gpflow/versions.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Personal package imports

In [4]:
from py_group_sequential_designs import boundaries as bd
from py_group_sequential_designs import feasibility_penalty as fp
from py_group_sequential_designs import format_boundaries_after_ask as fmt_bd
from py_group_sequential_designs import function_to_minimize as fn_min
from py_group_sequential_designs import generate_gpr_input as gen_input
from py_group_sequential_designs import simulate as sim
from py_group_sequential_designs import sample_size as ss

# Bayesian optimization workflow default values

In [5]:
# some set defaults
num_analyses = 3
target_alpha = 0.05
target_power = 0.9
important_diff_delta = 1
assumed_variance = 3

# to obtain mu (sample size at one stage)
# assume 1:1 randomization
group_ratio = 1

# Simulate initial points for GPR

## Create a helper function

In [6]:
# create a function that generates the points x and y that will be
# included in the design matrix X and Y
def generate_x_y(
        upper_bounds,
        lower_bounds,
        n_analyses,
        alt_hypothesis,
        variance,
        ratio,
        target_power,
        target_alpha,
        alpha_prime,
        beta_prime,
        n_power09):
    
    # 2. Generate the GPR input values
    # note that the input includes the sample size at power 0.9
    x = gen_input.generate_gpr_input(
        n_analyses = n_analyses,
        upper_bounds=upper_bounds,
        lower_bounds=lower_bounds,
        n_patients=n_power09)
    
    # 3. Generate maximum expected sample size and feasibility penalty
    max_ess_new = ss.max_ess(
        n_analyses=n_analyses,
        upper_bounds=upper_bounds,
        lower_bounds=lower_bounds,
        n_patients=n_power09)
    
    penalty = fp.feasibility_penalty(
        ratio=ratio,
        variance=variance,
        power=target_power,
        alpha=target_alpha,
        delta=important_diff_delta,
        beta_prime=beta_prime,
        alpha_prime=alpha_prime
    )
    
    
    # 4. Calculate the function value (GPR output)
    y = fn_min.function_to_minimize(max_ess_val=max_ess_new, penalty=penalty)

    return (np.array([x]), np.array([[y]]))

## Point 1

In [7]:
# simulate the trial design 
poc_simulation = bd.calculate_pocock_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    n_patients=20
)

# find the number of patients that achieves 90% power (beta 0.1)
# here we get beta_prime
poc_n_power09, poc_beta_prime = ss.find_sample_size(
    n_analyses = num_analyses,
    upper_bounds = poc_simulation[0],
    lower_bounds = poc_simulation[1],
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance
)

x1, y1 = generate_x_y(
    upper_bounds = poc_simulation[0],
    lower_bounds = poc_simulation[1],
    n_analyses = num_analyses,
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance,
    ratio = group_ratio,
    target_power = target_power,
    target_alpha = target_alpha,
    alpha_prime = poc_simulation[3],
    beta_prime = poc_beta_prime,
    n_power09 = poc_n_power09
)

## Point 2

In [8]:
# simulate the trial design 
of_simulation = bd.calculate_of_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    n_patients=20
)

# find the number of patients that achieves 90% power (beta 0.1) and beta_prime
of_n_power09, of_beta_prime = ss.find_sample_size(
    n_analyses = num_analyses,
    upper_bounds = of_simulation[0],
    lower_bounds = of_simulation[1],
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance
)

x2, y2 = generate_x_y(
    upper_bounds = of_simulation[0],
    lower_bounds = of_simulation[1],
    n_analyses = num_analyses,
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance,
    ratio = group_ratio,
    target_power = target_power,
    target_alpha = target_alpha,
    alpha_prime = of_simulation[3],
    beta_prime = of_beta_prime,
    n_power09 = of_n_power09
)

## Point 3

In [9]:
# simulate the trial design 
tri_simulation = bd.calculate_triangular_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    delta=important_diff_delta,
    n_patients=20
)

# find the number of patients that achieves 90% power (beta 0.1) and beta_prime
tri_n_power09, tri_beta_prime = ss.find_sample_size(
    n_analyses = num_analyses,
    upper_bounds = tri_simulation[0],
    lower_bounds = tri_simulation[1],
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance
)

x3, y3 = generate_x_y(
    upper_bounds = tri_simulation[0],
    lower_bounds = tri_simulation[1],
    n_analyses = num_analyses,
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance,
    ratio = group_ratio,
    target_power = target_power,
    target_alpha = target_alpha,
    alpha_prime = tri_simulation[3],
    beta_prime = tri_beta_prime,
    n_power09 = tri_n_power09
)

# Bayesian optimization loop

## Enter the initial points

In [10]:
design_matrix = np.concatenate((x1, x2, x3))
design_matrix

array([[ 1.99218570e+00,  1.99218570e+00,  1.99218570e+00,
        -1.99218570e+00, -1.99218570e+00,  1.99629200e+01],
       [ 2.96112264e+00,  2.09382990e+00,  1.70960495e+00,
        -2.96112264e+00, -2.09382990e+00,  1.75537765e+01],
       [ 2.11957748e+00,  1.87345951e+00,  1.83560794e+00,
         6.28553399e-16,  1.12407571e+00,  2.13211763e+01]])

In [11]:
output_vals = np.concatenate((y1, y2, y3))
output_vals

array([[57.419834  ],
       [51.94949786],
       [45.46818138]])

## Center and scale

In [12]:
def normalize_forward(data, mean=None, std=None):
    if (mean is None) and (std is None):
        mean = np.mean(data)
        std = np.std(data)
    
    return (
        (data - mean) / std,
        mean,
        std
    )

In [13]:
def normalize_backward(norm_data, mean, std):
    return (norm_data * std) + mean

### Test the functions

In [14]:
test_data, mean, std = normalize_forward(design_matrix)
test_data

array([[-0.25656921, -0.25656921, -0.25656921, -0.80367424, -0.80367424,
         2.21104191],
       [-0.1235218 , -0.24261216, -0.29537115, -0.93672165, -0.81763129,
         1.88023578],
       [-0.23907669, -0.27287183, -0.27806933, -0.53012173, -0.37577179,
         2.39754785]])

In [15]:
normalize_backward(test_data, mean, std)

array([[ 1.99218570e+00,  1.99218570e+00,  1.99218570e+00,
        -1.99218570e+00, -1.99218570e+00,  1.99629200e+01],
       [ 2.96112264e+00,  2.09382990e+00,  1.70960495e+00,
        -2.96112264e+00, -2.09382990e+00,  1.75537765e+01],
       [ 2.11957748e+00,  1.87345951e+00,  1.83560794e+00,
         4.44089210e-16,  1.12407571e+00,  2.13211763e+01]])

## Normalize and loop

In [16]:
normed_design_matix, design_matrix_mean, design_matrix_std = normalize_forward(design_matrix)

In [17]:
normed_output_vals, output_vals_mean, output_vals_std = normalize_forward(output_vals)

In [18]:
def build_model(X, Y):
    
    kernel = gpflow.kernels.SquaredExponential()

    likelihood = gpflow.likelihoods.Gaussian()
        
    gpr = gpflow.models.GPR(
        data = (X, Y),
        kernel = kernel,
        likelihood = likelihood
    )

    gpflow.utilities.print_summary(gpr, fmt="notebook")
    
    return GaussianProcessRegression(gpr)

In [19]:
bayes_opt_model = build_model(X = normed_design_matix, Y = normed_output_vals)

name,class,transform,prior,trainable,shape,dtype,value
GPR.kernel.variance,Parameter,Softplus,,True,(),float64,1
GPR.kernel.lengthscales,Parameter,Softplus,,True,(),float64,1
GPR.likelihood.variance,Parameter,Softplus + Shift,,True,(),float64,1


In [20]:
# create a dataset that works well with trieste
initial_data = trieste.data.Dataset(
    query_points = normed_design_matix, 
    observations = normed_output_vals
)

In [21]:
# normalize the search space
x_search_space = [-20, -20, -20, 20, 20]
y_search_space = 1000

In [22]:
normalize_forward(x_search_space,
                  design_matrix_mean,
                  design_matrix_std)

(array([-3.27637693, -3.27637693, -3.27637693,  2.21613348,  2.21613348]),
 3.860688006245452,
 7.282644369405211)

In [23]:
normalize_forward(y_search_space,
                  output_vals_mean,
                  output_vals_std)

(194.1405159315543, 51.61250441452513, 4.885057047648086)

In [24]:
# create the search space using trieste Box function
search_space = Box(
    lower = [-4, -4, -4, -4, -4, 0], 
    upper = [3, 3, 3, 3, 3, 200]
)

In [25]:
initial_data

Dataset(query_points=array([[-0.25656921, -0.25656921, -0.25656921, -0.80367424, -0.80367424,
         2.21104191],
       [-0.1235218 , -0.24261216, -0.29537115, -0.93672165, -0.81763129,
         1.88023578],
       [-0.23907669, -0.27287183, -0.27806933, -0.53012173, -0.37577179,
         2.39754785]]), observations=array([[ 1.18879463],
       [ 0.06898455],
       [-1.25777918]]))

In [26]:
ask_tell = trieste.ask_tell_optimization.AskTellOptimizer(
    search_space = search_space,
    datasets = initial_data,
    models = bayes_opt_model
)

# Having trouble with the normalization sequence below

In [33]:
num_repeats = 2

for i in range(num_repeats):
    normed_results = ask_tell.ask()

    results_x = normalize_backward(
        normed_results[0][0:5],
        design_matrix_mean,
        design_matrix_std
    )    

    results_y = normalize_backward(
        normed_results[0][5],
        output_vals_mean,
        output_vals_std
    )

    unnormed_results = np.array(
        np.concatenate((results_x[0:5], [results_y]))
    )
    
    unnormed_new_inputs = fmt_bd.format_boundaries_after_ask(
        n_analyses = num_analyses,
        result = unnormed_results
    )

    print(unnormed_new_inputs)

    new_sim_trial = sim.group_sequential_designs(
        n_analyses = num_analyses,
        upper_bounds = unnormed_new_inputs[0],
        lower_bounds = unnormed_new_inputs[1],
        n_patients = unnormed_new_inputs[2],
        null_hypothesis = 0,
        alt_hypothesis = important_diff_delta,
        variance = assumed_variance
    )

    new_x, new_y = generate_x_y(
        upper_bounds = unnormed_new_inputs[0],
        lower_bounds= unnormed_new_inputs[1],
        n_analyses = num_analyses,
        alt_hypothesis = important_diff_delta,
        variance = assumed_variance,
        ratio = group_ratio,
        target_power = target_power,
        target_alpha = target_alpha,
        alpha_prime = new_sim_trial[1],
        beta_prime = new_sim_trial[2],
        n_power09 = unnormed_new_inputs[2]
    )

    print(new_x, new_y)

    normed_new_x = normalize_forward(
        new_x,
        design_matrix_mean,
        design_matrix_std
    )

    normed_new_y = normalize_forward(
        new_y,
        output_vals_mean,
        output_vals_std
    )

    new_data = trieste.data.Dataset(
        query_points = normed_new_x, 
        observations = normed_new_y
    )

    ask_tell.tell(new_data=new_data)

IndexError: invalid index to scalar variable.

In [ ]:
results = ask_tell.ask()

In [ ]:
results

In order to obtain the new output, we need to use this above information to generate the penalty.

In [ ]:
new_inputs = fmt_bd.format_boundaries_after_ask(
    n_analyses = num_analyses,
    result = results
)

In [ ]:
new_sim_trial = sim.group_sequential_designs(
    n_analyses = num_analyses,
    upper_bounds = new_inputs[0],
    lower_bounds = new_inputs[1],
    n_patients = new_inputs[2],
    null_hypothesis = 0,
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance
)

In [ ]:
new_x, new_y = generate_x_y(
    upper_bounds = new_inputs[0],
    lower_bounds= new_inputs[1],
    n_analyses = num_analyses,
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance,
    ratio = group_ratio,
    target_power = target_power,
    target_alpha = target_alpha,
    alpha_prime = new_sim_trial[1],
    beta_prime = new_sim_trial[2],
    n_power09 = new_inputs[2]
)

In [ ]:
test

In [ ]:
# create a dataset that works well with trieste


In [ ]:
ask_tell.tell(new_data=new_data)

In [ ]:
ask_tell.ask()